## Problems
The energy dataframe seems to have a lot of Property names that are the same as the Address. Therefore telling us no info.    
The business dataframe doesn't have a full address, we need to make the column

In [ ]:
import sys

!{sys.executable} -m pip install pandas
%pip install pandas
import pandas as pd
pd.set_option("display.max_columns", None)
energy = pd.read_csv("2026-sandiego-covered-building-list-public-version.xlsx - 2026 SD CBL(Public version).csv")
energy.columns = energy.iloc[3]
energy = energy.iloc[4:].reset_index()
energy = energy.drop(columns=["index"])




zsh:1: no such file or directory: /Users/jadenwu/Desktop/Move
Note: you may need to restart the kernel to use updated packages.


We have several rows where the Propoerty Name and Address are the exact same, therefore rendering the Property Name to be completely useless

In [4]:
energy['Address 1'].isna().sum()
matches = energy[energy['Property Name'].str.lower()==energy['Address 1'].str.lower()]
bad_rows = len(matches)
print(f"{round(bad_rows/len(energy), 3)}% of rows don't have a property name")


0.41% of rows don't have a property name


In [23]:
business = pd.read_csv("sd_businesses_active_datasd.csv")
import re

def zip_code(x):
    x = "" if pd.isna(x) else str(x).strip()
    m = re.match(r"(\d{5})", x)
    return m.group(1) if m else ""

def norm_addr(x):
    x = "" if pd.isna(x) else str(x).upper()
    x = re.sub(r"[^\w\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

energy['join_key'] = energy["Address 1"].map(norm_addr) + "|" + energy["Postal Code"].map(zip_code)

business["address_full"] = (
    business["address_no"].fillna("").astype(str) + " " +
    business["address_pd"].fillna("").astype(str) + " " +
    business["address_road"].fillna("").astype(str) + " " +
    business["address_sfx"].fillna("").astype(str)
).str.replace(r"\s+", " ", regex=True).str.strip()

business["join_key"] = business["address_full"].map(norm_addr) + "|" + business["address_zip"].map(zip_code)

energy_business = energy.merge(
    business,
    on="join_key",
    how="left",
    suffixes=("_energy", "_biz"),
    indicator=True
)

print(energy_business["_merge"].value_counts())

_merge
both          3697
left_only     1338
right_only       0
Name: count, dtype: int64


/var/folders/xg/pjqjkzbx1lz9fgmz_njxvl6w0000gn/T/ipykernel_85903/463718827.py:1: DtypeWarning: Columns (0: address_po_box) have mixed types. Specify dtype option on import or set low_memory=False.
  business = pd.read_csv("sd_businesses_active_datasd.csv")


In [21]:
business['address_no_fraction'].isna().sum()/len(business['address_no_fraction'])

np.float64(0.9960191743104049)

In [24]:
id_col = "Verified Standard ID - City/Town ID"

matched = merged[merged["_merge"] == "both"]
unique_energy_matched = matched[id_col].nunique()
extra_dupe_rows = len(matched) - unique_energy_matched

print("unique matched energy rows:", unique_energy_matched)
print("extra duplicate rows:", extra_dupe_rows)

NameError: name 'merged' is not defined